<a href="https://colab.research.google.com/github/asheldrick-research/ecsm-framework/blob/main/ECSM_NG22R_E2_W1_Baryonic_Ledger_Lock_COLAB_PATCHED_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# ECSM NG22R-E2 — W1 baryonic-ledger construction and lock

This notebook is the final **pre-score** stage of the locked WALLABY transfer.

It accepts the E1 ZIP, verifies its registry and sealed-ledger hashes, then performs only
independent baryonic reconstruction:

- resolves official WALLABY Hubble distances (`dist_h`, Mpc);
- amends W1 centres to `RA_model/DEC_model` before image acquisition;
- downloads official AllWISE W1 intensity and uncertainty cutouts from IRSA SIA v2;
- extracts robust fixed-geometry stellar profiles without using `Vrot_model`;
- selects disk-only or disk-plus-bulge using the preregistered ΔBIC > 10 rule;
- uses the released face-on H I profile and the fixed helium factor 1.33;
- calculates finite-thickness gas and stellar unit-mass-to-light rotation contributions;
- creates and hashes a baryonic ledger;
- does **not** calculate ECSM response, residuals, chi-square, or any comparison with the
  sealed observed rotation curve.

The final light package is saved to Google Drive. W1 FITS files remain in a separate
Drive folder with a complete URL and SHA-256 manifest.


## Patch v2 — pre-score acquisition repair

The original E2 notebook modified the SIA `access_url` with unsupported guessed
cutout parameters and derived uncertainty filenames rather than using the exact
products returned by IRSA. That caused all W1 acquisitions to fail and left an
empty audit table.

This version downloads the exact intensity, uncertainty and coverage URLs returned
by IRSA, creates the cutouts locally, records all failures safely, and updates the
AllWISE Atlas W1 PSF audit value from 6.1 arcsec to the documented approximate
8.3 arcsec. No rotation curve has been scored and no ECSM parameter has changed.


In [4]:
#@title 1. Upload E1, mount Drive, install dependencies
!pip -q install astropy scipy pyvo

from pathlib import Path
import io, os, re, json, math, time, shutil, zipfile, hashlib
import numpy as np
import pandas as pd
import requests
from scipy.optimize import least_squares
from scipy.special import gammainc, gamma
from astropy.io import fits
from astropy.wcs import WCS
from astropy.nddata import Cutout2D
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.stats import sigma_clip
from google.colab import files, drive

drive.mount("/content/drive")

uploaded = files.upload()
zip_candidates = [
    Path(name) for name in uploaded
    if name.lower().endswith(".zip") and "NG22R_E1" in name
]
assert len(zip_candidates) == 1, (
    "Upload exactly one NG22R_E1_WALLABY_OFFICIAL_INPUT_LOCK.zip file."
)
E1_ZIP = zip_candidates[0]

ROOT = Path("/content/ng22r_e2_baryonic_lock")
E1 = ROOT / "e1"
W1 = ROOT / "w1_fits"
OUT = ROOT / "outputs"
AUDIT = OUT / "audit"
TABLES = OUT / "tables"
REGISTRY = OUT / "registry"
FIGURES = OUT / "figures"

if ROOT.exists():
    shutil.rmtree(ROOT)
for p in [E1, W1, AUDIT, TABLES, REGISTRY, FIGURES]:
    p.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(E1_ZIP) as z:
    z.extractall(E1)

PARENT_MODEL_HASH = "154dfb6daab67756d88d526118a9773075d6d824f0431405551c26f07451fe5a"
E0_HASH = "3d21f977bd27d961b57f2692c4a0f3aba1ab0b45d88bee1b9f05245ece73e9c2"
EXPECTED_E1_HASH = "e15440878d2152b6bd77b5c53602ad86036302b06472fe56b6f63f5fac207870"

# Locked photometric/dynamical reconstruction choices.
WISE_COLLECTION = "wise_allwise"
WISE_BAND_RANGE = "3.0e-6 4.0e-6"
WISE_W1_SOLAR_MAG_VEGA = 3.24
WISE_W1_PSF_FWHM_ARCSEC = 8.3
HELIUM_FACTOR = 1.33
PRIMARY_YDISK = 0.5
PRIMARY_YBULGE = 0.7
DISK_THICKNESS_OVER_SCALE_LENGTH = 0.20
MIN_STELLAR_SCALE_HEIGHT_KPC = 0.10
GAS_SCALE_HEIGHT_KPC = 0.10
BULGE_DELTA_BIC_THRESHOLD = 10.0
PROFILE_BIN_ARCSEC = 6.0
W1_BACKGROUND_INNER_RMAX = 2.2
W1_BACKGROUND_OUTER_RMAX = 2.8
W1_PROFILE_OUTER_RMAX = 1.5
MIN_PROFILE_BINS = 8

# Locked data-adequacy gate, declared before external scoring.
MIN_SUCCESS_FRACTION_PER_SAMPLE = 0.80
MIN_DR2_SUCCESS = 12
MIN_DR1_SUCCESS = 19

G_KPC_KMS2_MSUN = 4.30091e-6
KPC_TO_M = 3.0856775814913673e19
PC_PER_KPC = 1000.0
SIA_ENDPOINT = "https://irsa.ipac.caltech.edu/SIA"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Saving NG22R_E1_WALLABY_OFFICIAL_INPUT_LOCK (2).zip to NG22R_E1_WALLABY_OFFICIAL_INPUT_LOCK (2).zip


In [5]:

#@title 2. Verify E1 hashes and create the formal pre-score amendment

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

e1_registry_path = E1 / "registry/ng22r_e1_official_input_lock_registry.json"
e1_registry = json.loads(e1_registry_path.read_text())
claimed = e1_registry["E1_registry_sha256"]
canonical_obj = dict(e1_registry)
canonical_obj.pop("E1_registry_sha256")
calculated = hashlib.sha256(
    json.dumps(canonical_obj, sort_keys=True, separators=(",", ":")).encode()
).hexdigest()

assert claimed == EXPECTED_E1_HASH
assert calculated == EXPECTED_E1_HASH
assert e1_registry["parent_model_freeze_sha256"] == PARENT_MODEL_HASH
assert e1_registry["parent_E0_transfer_protocol_sha256"] == E0_HASH

for record in e1_registry["raw_catalogues"]:
    path = E1 / "raw" / record["filename"]
    assert sha256_file(path) == record["sha256"]

sealed_path = E1 / e1_registry["sealed_observed_rotation_ledger"]["path"]
assert sha256_file(sealed_path) == e1_registry["sealed_observed_rotation_ledger"]["sha256"]

# The sealed CSV is deliberately not read.
sealed_hash = sha256_file(sealed_path)

geometry = pd.read_csv(E1 / "parsed/wallaby_angular_geometry_and_w1_targets.csv")
quality = pd.read_csv(E1 / "audit/wallaby_preregistered_quality_gate.csv")
dr1_kin = pd.read_csv(E1 / "raw/dr1_kinematic_catalogue.csv")
dr2_kin = pd.read_csv(E1 / "raw/dr2_kinematic_catalogue.csv")
dr1_src = pd.read_csv(E1 / "raw/dr1_source_catalogue.csv")
dr2_src = pd.read_csv(E1 / "raw/dr2_source_catalogue.csv")

def norm_name(value):
    return re.sub(r"[^A-Z0-9]", "", str(value).upper())

for frame in [dr1_kin, dr2_kin, dr1_src, dr2_src]:
    frame["_name_norm"] = frame["name"].map(norm_name)

targets = geometry[geometry["primary_quality_pass"]].copy()
targets["_name_norm"] = targets["galaxy"].map(norm_name)

amendment_rows = []
for idx, target in targets.iterrows():
    if target["sample"] == "DR2_phase2_primary":
        kin = dr2_kin[dr2_kin["_name_norm"] == target["_name_norm"]]
        src = dr2_src[dr2_src["_name_norm"] == target["_name_norm"]]
    else:
        kin = dr1_kin[dr1_kin["_name_norm"] == target["_name_norm"]]
        src = dr1_src[dr1_src["_name_norm"] == target["_name_norm"]]
    assert len(kin) >= 1 and len(src) >= 1
    kin = kin.iloc[0]
    src = src.iloc[0]

    ra_model = pd.to_numeric(kin.get("RA_model"), errors="coerce")
    dec_model = pd.to_numeric(kin.get("DEC_model"), errors="coerce")
    if not np.isfinite(ra_model):
        ra_model = target["ra_deg"]
    if not np.isfinite(dec_model):
        dec_model = target["dec_deg"]

    old_coord = SkyCoord(target["ra_deg"] * u.deg, target["dec_deg"] * u.deg)
    new_coord = SkyCoord(float(ra_model) * u.deg, float(dec_model) * u.deg)

    targets.loc[idx, "w1_ra_deg"] = float(ra_model)
    targets.loc[idx, "w1_dec_deg"] = float(dec_model)
    targets.loc[idx, "distance_mpc"] = float(src["dist_h"])
    targets.loc[idx, "model_center_offset_arcsec"] = float(old_coord.separation(new_coord).arcsec)

    amendment_rows.append({
        "source_product_id": target["source_product_id"],
        "galaxy": target["galaxy"],
        "sample": target["sample"],
        "E1_ra_deg": target["ra_deg"],
        "E1_dec_deg": target["dec_deg"],
        "amended_RA_model_deg": float(ra_model),
        "amended_DEC_model_deg": float(dec_model),
        "offset_arcsec": float(old_coord.separation(new_coord).arcsec),
        "distance_mpc_dist_h": float(src["dist_h"]),
        "amendment_reason": (
            "E0 preregistration specified RA_model/DEC_model where available; "
            "E1 ledger had stored general catalogue ra/dec."
        ),
        "rotation_values_used": False,
    })

amendment = pd.DataFrame(amendment_rows)
amendment.to_csv(AUDIT / "E1A_pre_score_coordinate_and_manifest_amendment.csv", index=False)
targets.to_csv(TABLES / "wallaby_E2_W1_targets_locked.csv", index=False)

print("E1 integrity verified.")
print("Sealed rotation-ledger SHA-256:", sealed_hash)
display(
    targets.groupby("sample", as_index=False)
    .agg(N_targets=("galaxy", "nunique"),
         median_center_amendment_arcsec=("model_center_offset_arcsec", "median"),
         median_distance_mpc=("distance_mpc", "median"))
)

E1 integrity verified.
Sealed rotation-ledger SHA-256: a4388cf03eb8ab19d8e9dc038577138dccf380a6f23ea5c8f445e2b7e539906d


,sample,N_targets,median_center_amendment_arcsec,median_distance_mpc
0,DR1_phase1_replication,24,6.022335,23.708113
1,DR2_phase2_primary,15,9.801044,36.853249


In [6]:
%pip install -q pyvo

In [7]:
import pyvo as vo
print("pyvo installed:", vo.__version__)

pyvo installed: 1.9.1


In [8]:
#@title 3. Acquire official AllWISE W1 products — FIXED v7

import io
import math
import re
import shutil
import time

import numpy as np
import pandas as pd
import requests

from astropy.io import fits


# -------------------------------------------------------------------
# Official IRSA AllWISE endpoints
# -------------------------------------------------------------------

SEARCH_URL = (
    "https://irsa.ipac.caltech.edu/ibe/search/"
    "wise/allwise/p3am_cdd"
)

DATA_ROOT = (
    "https://irsa.ipac.caltech.edu/ibe/data/"
    "wise/allwise/p3am_cdd"
)

WISE_PIXEL_SCALE_ARCSEC = 1.375


# Remove files from previous failed attempts.
if W1.exists():
    shutil.rmtree(W1)

W1.mkdir(parents=True, exist_ok=True)


# -------------------------------------------------------------------
# Metadata query
# -------------------------------------------------------------------

def query_allwise_tiles(ra_deg, dec_deg):
    """
    Return every AllWISE W1 coadd containing the requested position.

    Only coadd_id and band are requested because those are the
    documented identifiers for this metadata table.
    """

    response = requests.get(
        SEARCH_URL,
        params={
            "POS": f"{float(ra_deg):.8f},{float(dec_deg):.8f}",
            "where": "band=1",
            "columns": "coadd_id,band",
            "ct": "CSV",
        },
        timeout=180,
        headers={
            "User-Agent": "ECSM-NG22R-E2/7.0",
            "Accept": "text/csv,text/plain,*/*",
        },
    )

    response.raise_for_status()

    if len(response.content) < 20:
        raise RuntimeError(
            "IRSA returned an empty metadata response."
        )

    response_start = response.text.lstrip()[:2000]

    if (
        response_start.startswith("<?xml")
        or "<VOTABLE" in response_start
    ):
        raise RuntimeError(
            "IRSA returned an XML error instead of CSV. "
            f"Request URL: {response.url}. "
            f"Response begins: {response.text[:1200]}"
        )

    try:
        table = pd.read_csv(
            io.BytesIO(response.content)
        )

    except Exception as exc:
        raise RuntimeError(
            "Could not parse IRSA metadata CSV. "
            f"{type(exc).__name__}: {exc}. "
            f"Response begins: {response.text[:1000]}"
        ) from exc

    if table.empty:
        raise RuntimeError(
            "No AllWISE W1 Atlas tile covers this position."
        )

    table.columns = [
        str(column).strip()
        for column in table.columns
    ]

    if "coadd_id" not in table.columns:
        raise RuntimeError(
            "IRSA response does not contain coadd_id. "
            "Returned columns: "
            + ", ".join(table.columns)
        )

    table["coadd_id"] = (
        table["coadd_id"]
        .astype(str)
        .str.strip()
    )

    table = table[
        table["coadd_id"].notna()
        & (table["coadd_id"] != "")
        & (
            table["coadd_id"]
            .str.lower()
            != "nan"
        )
    ].copy()

    if "band" in table.columns:
        band = pd.to_numeric(
            table["band"],
            errors="coerce",
        )

        w1_rows = table[
            band == 1
        ].copy()

        if not w1_rows.empty:
            table = w1_rows

    coadd_ids = sorted(
        table["coadd_id"]
        .drop_duplicates()
        .tolist()
    )

    if not coadd_ids:
        raise RuntimeError(
            "IRSA returned no valid W1 coadd identifiers."
        )

    return coadd_ids


# -------------------------------------------------------------------
# Product URLs
# -------------------------------------------------------------------

def build_product_url(
    coadd_id,
    product,
    ra_deg,
    dec_deg,
    width_arcsec,
):
    """
    Construct an official AllWISE Atlas cutout URL.
    """

    coadd_group = coadd_id[:2]
    coadd_ra = coadd_id[:4]

    if product == "int":
        filename = (
            f"{coadd_id}-w1-int-3.fits"
        )

    elif product == "unc":
        filename = (
            f"{coadd_id}-w1-unc-3.fits.gz"
        )

    elif product == "cov":
        filename = (
            f"{coadd_id}-w1-cov-3.fits.gz"
        )

    else:
        raise ValueError(
            f"Unknown AllWISE product: {product}"
        )

    size_pixels = int(
        np.clip(
            math.ceil(
                float(width_arcsec)
                / WISE_PIXEL_SCALE_ARCSEC
            ),
            64,
            2000,
        )
    )

    base_url = (
        f"{DATA_ROOT}/"
        f"{coadd_group}/"
        f"{coadd_ra}/"
        f"{coadd_id}/"
        f"{filename}"
    )

    return (
        f"{base_url}"
        f"?center="
        f"{float(ra_deg):.8f},"
        f"{float(dec_deg):.8f}"
        f"&size={size_pixels}pix"
        f"&gzip=false"
    )


# -------------------------------------------------------------------
# FITS download and validation
# -------------------------------------------------------------------

def download_and_validate(
    url,
    output_path,
    retries=4,
):
    """
    Download and validate one FITS cutout.
    """

    errors = []

    for attempt in range(
        1,
        retries + 1,
    ):
        try:
            response = requests.get(
                url,
                timeout=300,
                headers={
                    "User-Agent": "ECSM-NG22R-E2/7.0",
                    "Accept": (
                        "application/fits,"
                        "application/octet-stream,"
                        "*/*"
                    ),
                },
            )

            response.raise_for_status()

            if len(response.content) < 1000:
                raise RuntimeError(
                    f"Response was only "
                    f"{len(response.content)} bytes. "
                    f"Content-Type: "
                    f"{response.headers.get('content-type', '')}. "
                    f"Response begins: "
                    f"{response.text[:300]}"
                )

            output_path.write_bytes(
                response.content
            )

            with fits.open(
                output_path,
                memmap=False,
            ) as hdul:

                image = hdul[0].data

                if image is None:
                    raise RuntimeError(
                        "Downloaded FITS contains no image."
                    )

                image = np.squeeze(
                    np.asarray(image)
                )

                if image.ndim != 2:
                    raise RuntimeError(
                        "Unexpected FITS image shape: "
                        f"{image.shape}"
                    )

                if image.size == 0:
                    raise RuntimeError(
                        "Downloaded FITS image is empty."
                    )

                finite_fraction = float(
                    np.isfinite(image).mean()
                )

                if finite_fraction < 0.05:
                    raise RuntimeError(
                        "FITS cutout contains fewer than "
                        "5% finite pixels."
                    )

            return

        except Exception as exc:
            errors.append(
                f"attempt {attempt}: "
                f"{type(exc).__name__}: {exc}"
            )

            if output_path.exists():
                output_path.unlink()

            time.sleep(
                2 * attempt
            )

    raise RuntimeError(
        " | ".join(errors)
    )


# -------------------------------------------------------------------
# Try overlapping tiles
# -------------------------------------------------------------------

def acquire_target(
    row,
    intensity_path,
    uncertainty_path,
    coverage_path,
):
    """
    Try each overlapping W1 coadd until valid intensity and
    uncertainty cutouts are obtained.
    """

    coadd_ids = query_allwise_tiles(
        row.w1_ra_deg,
        row.w1_dec_deg,
    )

    tile_errors = []

    for coadd_id in coadd_ids:

        intensity_url = build_product_url(
            coadd_id=coadd_id,
            product="int",
            ra_deg=row.w1_ra_deg,
            dec_deg=row.w1_dec_deg,
            width_arcsec=(
                row.w1_cutout_width_arcsec
            ),
        )

        uncertainty_url = build_product_url(
            coadd_id=coadd_id,
            product="unc",
            ra_deg=row.w1_ra_deg,
            dec_deg=row.w1_dec_deg,
            width_arcsec=(
                row.w1_cutout_width_arcsec
            ),
        )

        coverage_url = build_product_url(
            coadd_id=coadd_id,
            product="cov",
            ra_deg=row.w1_ra_deg,
            dec_deg=row.w1_dec_deg,
            width_arcsec=(
                row.w1_cutout_width_arcsec
            ),
        )

        try:
            download_and_validate(
                intensity_url,
                intensity_path,
            )

            download_and_validate(
                uncertainty_url,
                uncertainty_path,
            )

            # Coverage is optional.
            try:
                download_and_validate(
                    coverage_url,
                    coverage_path,
                )

            except Exception:
                if coverage_path.exists():
                    coverage_path.unlink()

            return {
                "coadd_id": coadd_id,
                "intensity_url": intensity_url,
                "uncertainty_url": uncertainty_url,
                "coverage_url": coverage_url,
            }

        except Exception as exc:
            tile_errors.append(
                f"{coadd_id}: "
                f"{type(exc).__name__}: {exc}"
            )

            for path in [
                intensity_path,
                uncertainty_path,
                coverage_path,
            ]:
                if path.exists():
                    path.unlink()

    raise RuntimeError(
        "All overlapping W1 tiles failed. "
        + " || ".join(tile_errors)
    )


# -------------------------------------------------------------------
# Acquire preregistered target set
# -------------------------------------------------------------------

acquisition_rows = []

total_targets = len(targets)


for number, row in enumerate(
    targets.itertuples(index=False),
    start=1,
):

    safe_name = re.sub(
        r"[^A-Za-z0-9_-]",
        "_",
        row.source_product_id,
    )

    intensity_path = (
        W1
        / f"{safe_name}_W1_int.fits"
    )

    uncertainty_path = (
        W1
        / f"{safe_name}_W1_unc.fits"
    )

    coverage_path = (
        W1
        / f"{safe_name}_W1_cov.fits"
    )

    status = "success"
    error_text = ""
    coadd_id = ""

    intensity_url = ""
    uncertainty_url = ""
    coverage_url = ""

    print(
        f"[{number}/{total_targets}]",
        row.sample,
        row.galaxy,
        "...",
        flush=True,
    )

    try:
        result = acquire_target(
            row=row,
            intensity_path=intensity_path,
            uncertainty_path=uncertainty_path,
            coverage_path=coverage_path,
        )

        coadd_id = result["coadd_id"]
        intensity_url = result["intensity_url"]
        uncertainty_url = result["uncertainty_url"]
        coverage_url = result["coverage_url"]

        print(
            "   success:",
            coadd_id,
            flush=True,
        )

    except Exception as exc:
        status = "failed"

        error_text = (
            f"{type(exc).__name__}: {exc}"
        )

        for path in [
            intensity_path,
            uncertainty_path,
            coverage_path,
        ]:
            if path.exists():
                path.unlink()

        print(
            "   FAILED:",
            error_text[:1200],
            flush=True,
        )

    acquisition_rows.append(
        {
            "source_product_id": (
                row.source_product_id
            ),
            "galaxy": row.galaxy,
            "sample": row.sample,
            "coadd_id": coadd_id,
            "status": status,
            "error": error_text,
            "intensity_access_url": (
                intensity_url
            ),
            "uncertainty_access_url": (
                uncertainty_url
            ),
            "coverage_access_url": (
                coverage_url
            ),
            "intensity_file": (
                intensity_path.name
                if intensity_path.exists()
                else ""
            ),
            "uncertainty_file": (
                uncertainty_path.name
                if uncertainty_path.exists()
                else ""
            ),
            "coverage_file": (
                coverage_path.name
                if coverage_path.exists()
                else ""
            ),
            "intensity_sha256": (
                sha256_file(
                    intensity_path
                )
                if intensity_path.exists()
                else ""
            ),
            "uncertainty_sha256": (
                sha256_file(
                    uncertainty_path
                )
                if uncertainty_path.exists()
                else ""
            ),
            "coverage_sha256": (
                sha256_file(
                    coverage_path
                )
                if coverage_path.exists()
                else ""
            ),
        }
    )


# -------------------------------------------------------------------
# Save acquisition audit
# -------------------------------------------------------------------

acquisition = pd.DataFrame(
    acquisition_rows,
    columns=[
        "source_product_id",
        "galaxy",
        "sample",
        "coadd_id",
        "status",
        "error",
        "intensity_access_url",
        "uncertainty_access_url",
        "coverage_access_url",
        "intensity_file",
        "uncertainty_file",
        "coverage_file",
        "intensity_sha256",
        "uncertainty_sha256",
        "coverage_sha256",
    ],
)

acquisition.to_csv(
    AUDIT
    / "allwise_W1_acquisition_manifest.csv",
    index=False,
)

summary = (
    acquisition
    .groupby(
        ["sample", "status"]
    )
    .size()
    .reset_index(name="N")
)

display(summary)


successful_count = int(
    (
        acquisition["status"]
        == "success"
    ).sum()
)

failed_count = int(
    (
        acquisition["status"]
        == "failed"
    ).sum()
)

print(
    "Successful W1 acquisitions:",
    successful_count,
)

print(
    "Failed W1 acquisitions:",
    failed_count,
)


if successful_count == 0:

    display(
        acquisition[
            [
                "sample",
                "galaxy",
                "error",
            ]
        ].head(10)
    )

    raise RuntimeError(
        "No W1 acquisitions succeeded. "
        "The first ten exact errors are displayed above."
    )

[1/39] DR2_phase2_primary WALLABY J094524-480828 ...
   success: 1462m485_ac51
[2/39] DR2_phase2_primary WALLABY J100916-431959 ...
   success: 1519m440_ac51
[3/39] DR2_phase2_primary WALLABY J101655-485238 ...
   success: 1552m485_ac51
[4/39] DR2_phase2_primary WALLABY J125548+041805 ...
   success: 1944p045_ac51
[5/39] DR2_phase2_primary WALLABY J125549+040049 ...
   success: 1944p045_ac51
[6/39] DR2_phase2_primary WALLABY J125956-192430 ...
   success: 1952m197_ac51
[7/39] DR2_phase2_primary WALLABY J130213-145817 ...
   success: 1956m152_ac51
[8/39] DR2_phase2_primary WALLABY J130314-172514 ...
   success: 1950m182_ac51
[9/39] DR2_phase2_primary WALLABY J130415-102023 ...
   success: 1953m107_ac51
[10/39] DR2_phase2_primary WALLABY J130618-173039 ...
   success: 1966m182_ac51
[11/39] DR2_phase2_primary WALLABY J130943-163617 ...
   success: 1980m167_ac51
[12/39] DR2_phase2_primary WALLABY J131237-142630 ...
   success: 1987m152_ac51
[13/39] DR2_phase2_primary WALLABY J131658-163757

,sample,status,N
0,DR1_phase1_replication,success,24
1,DR2_phase2_primary,success,15


Successful W1 acquisitions: 39
Failed W1 acquisitions: 0


In [9]:
#@title 4. Extract fixed-geometry W1 profiles and build gas/stellar gravity ledgers — FIXED

import math
import re

import numpy as np
import pandas as pd

import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.stats import sigma_clip
from astropy.wcs import WCS
from astropy.wcs.utils import wcs_to_celestial_frame

from scipy.optimize import least_squares
from scipy.special import gamma, gammainc


# ===================================================================
# Catalogue-array parsing and exact kinematic-product selection
# ===================================================================

def parse_array(value):
    """
    Parse WALLABY catalogue array fields safely.

    Handles strings, lists, tuples, NumPy arrays and scalar values.
    """

    if value is None or np.ma.is_masked(value):
        return np.array([], dtype=float)

    if isinstance(
        value,
        (
            list,
            tuple,
            np.ndarray,
            pd.Series,
        ),
    ):
        try:
            array = np.asarray(
                value,
                dtype=float,
            ).ravel()

            return array[
                np.isfinite(array)
            ]

        except Exception:
            value = str(value)

    text = str(value).strip()

    if text.lower() in {
        "",
        "nan",
        "none",
        "--",
        "[]",
    }:
        return np.array([], dtype=float)

    tokens = re.findall(
        r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)"
        r"(?:[eE][-+]?\d+)?",
        text,
    )

    if not tokens:
        return np.array([], dtype=float)

    array = np.asarray(
        [
            float(token)
            for token in tokens
        ],
        dtype=float,
    )

    return array[
        np.isfinite(array)
    ]


def normalize_release_name(value):
    return re.sub(
        r"[^A-Z0-9]",
        "",
        str(value).upper(),
    )


def get_kinematic_row(target):
    """
    Retrieve the exact locked WALLABY kinematic product.

    Duplicate galaxy names are resolved using team_release_kin.
    """

    frame = (
        dr2_kin
        if target["sample"]
        == "DR2_phase2_primary"
        else dr1_kin
    )

    name_matches = frame[
        frame["_name_norm"]
        == target["_name_norm"]
    ].copy()

    if len(name_matches) == 0:
        raise RuntimeError(
            "No kinematic catalogue row found for "
            f"{target['galaxy']}."
        )

    locked_release = normalize_release_name(
        target["team_release_kin"]
    )

    release_matches = name_matches[
        name_matches[
            "team_release_kin"
        ].map(
            normalize_release_name
        )
        == locked_release
    ].copy()

    if len(release_matches) == 1:
        return release_matches.iloc[0]

    if len(release_matches) > 1:
        raise RuntimeError(
            "Multiple rows matched both galaxy and locked "
            f"team release for {target['galaxy']}."
        )

    if len(name_matches) == 1:
        return name_matches.iloc[0]

    available = sorted(
        name_matches[
            "team_release_kin"
        ].astype(str).unique()
    )

    raise RuntimeError(
        "No exact team-release match for "
        f"{target['galaxy']}. "
        f"Locked release: {target['team_release_kin']}. "
        f"Available releases: {available}"
    )


# ===================================================================
# Correct target centres using the exact locked kinematic release
# ===================================================================

corrected_amendment_rows = []
centre_correction_count = 0


for target_index, target_row in targets.iterrows():

    exact_kinematic_row = get_kinematic_row(
        target_row
    )

    exact_ra = pd.to_numeric(
        exact_kinematic_row.get(
            "RA_model"
        ),
        errors="coerce",
    )

    exact_dec = pd.to_numeric(
        exact_kinematic_row.get(
            "DEC_model"
        ),
        errors="coerce",
    )

    if not np.isfinite(exact_ra):
        exact_ra = float(
            target_row["ra_deg"]
        )

    if not np.isfinite(exact_dec):
        exact_dec = float(
            target_row["dec_deg"]
        )

    catalogue_coordinate = SkyCoord(
        float(
            target_row["ra_deg"]
        ) * u.deg,
        float(
            target_row["dec_deg"]
        ) * u.deg,
        frame="icrs",
    )

    previous_w1_coordinate = SkyCoord(
        float(
            target_row["w1_ra_deg"]
        ) * u.deg,
        float(
            target_row["w1_dec_deg"]
        ) * u.deg,
        frame="icrs",
    )

    exact_coordinate = SkyCoord(
        float(exact_ra) * u.deg,
        float(exact_dec) * u.deg,
        frame="icrs",
    )

    acquisition_to_exact_offset = float(
        previous_w1_coordinate
        .separation(
            exact_coordinate
        )
        .arcsec
    )

    if acquisition_to_exact_offset > 1e-6:
        centre_correction_count += 1

    targets.loc[
        target_index,
        "w1_ra_deg",
    ] = float(exact_ra)

    targets.loc[
        target_index,
        "w1_dec_deg",
    ] = float(exact_dec)

    targets.loc[
        target_index,
        "model_center_offset_arcsec",
    ] = float(
        catalogue_coordinate
        .separation(
            exact_coordinate
        )
        .arcsec
    )

    corrected_amendment_rows.append(
        {
            "source_product_id": (
                target_row[
                    "source_product_id"
                ]
            ),
            "galaxy": target_row[
                "galaxy"
            ],
            "sample": target_row[
                "sample"
            ],
            "team_release_kin": (
                target_row[
                    "team_release_kin"
                ]
            ),
            "E1_ra_deg": float(
                target_row["ra_deg"]
            ),
            "E1_dec_deg": float(
                target_row["dec_deg"]
            ),
            "previous_acquisition_RA_deg": float(
                target_row["w1_ra_deg"]
            ),
            "previous_acquisition_DEC_deg": float(
                target_row["w1_dec_deg"]
            ),
            "amended_RA_model_deg": float(
                exact_ra
            ),
            "amended_DEC_model_deg": float(
                exact_dec
            ),
            "offset_arcsec": float(
                catalogue_coordinate
                .separation(
                    exact_coordinate
                )
                .arcsec
            ),
            "acquisition_to_exact_offset_arcsec": (
                acquisition_to_exact_offset
            ),
            "distance_mpc_dist_h": float(
                target_row[
                    "distance_mpc"
                ]
            ),
            "amendment_reason": (
                "RA_model and DEC_model selected from the exact "
                "locked team_release_kin before W1 profile extraction."
            ),
            "rotation_values_used": False,
        }
    )


corrected_amendment = pd.DataFrame(
    corrected_amendment_rows
)

corrected_amendment.to_csv(
    AUDIT
    / "E1A_pre_score_coordinate_and_manifest_amendment.csv",
    index=False,
)

targets.to_csv(
    TABLES
    / "wallaby_E2_W1_targets_locked.csv",
    index=False,
)

print(
    "Exact-release kinematic matching enabled."
)

print(
    "W1 profile centres corrected:",
    centre_correction_count,
)

print(
    "Observed Vrot values accessed:",
    False,
)


# ===================================================================
# WCS and elliptical-profile functions
# ===================================================================

def local_tangent_offsets_arcsec(
    wcs,
    shape,
    ra,
    dec,
):
    """
    Calculate local east/north offsets for every image pixel.

    All coordinates are transformed to the same WCS celestial frame.
    """

    y, x = np.indices(
        shape,
        dtype=float,
    )

    wcs_frame = wcs_to_celestial_frame(
        wcs
    )

    target = SkyCoord(
        ra=float(ra) * u.deg,
        dec=float(dec) * u.deg,
        frame="icrs",
    ).transform_to(
        wcs_frame
    )

    x0, y0 = wcs.world_to_pixel(
        target
    )

    centre = wcs.pixel_to_world(
        x0,
        y0,
    )

    pixel_x = wcs.pixel_to_world(
        x0 + 1.0,
        y0,
    ).transform_to(
        centre.frame
    )

    pixel_y = wcs.pixel_to_world(
        x0,
        y0 + 1.0,
    ).transform_to(
        centre.frame
    )

    east_x, north_x = (
        centre.spherical_offsets_to(
            pixel_x
        )
    )

    east_y, north_y = (
        centre.spherical_offsets_to(
            pixel_y
        )
    )

    east = (
        (x - x0)
        * east_x.to_value(
            u.arcsec
        )
        + (y - y0)
        * east_y.to_value(
            u.arcsec
        )
    )

    north = (
        (x - x0)
        * north_x.to_value(
            u.arcsec
        )
        + (y - y0)
        * north_y.to_value(
            u.arcsec
        )
    )

    return (
        east,
        north,
        float(x0),
        float(y0),
    )


def elliptical_radius(
    east,
    north,
    pa_deg,
    inc_deg,
):
    pa = np.deg2rad(
        float(pa_deg)
    )

    cos_inc = max(
        np.cos(
            np.deg2rad(
                float(inc_deg)
            )
        ),
        0.15,
    )

    major = (
        east * np.sin(pa)
        + north * np.cos(pa)
    )

    minor = (
        -east * np.cos(pa)
        + north * np.sin(pa)
    )

    return np.sqrt(
        major**2
        + (
            minor
            / cos_inc
        ) ** 2
    )


def robust_background_plane(
    data,
    east,
    north,
    mask,
):
    finite = (
        mask
        & np.isfinite(data)
        & np.isfinite(east)
        & np.isfinite(north)
    )

    if finite.sum() < 100:
        raise RuntimeError(
            "Insufficient outer pixels for background plane."
        )

    xx = east[finite]
    yy = north[finite]
    zz = data[finite]

    keep = np.ones(
        len(zz),
        dtype=bool,
    )

    coefficients = np.zeros(
        3,
        dtype=float,
    )

    for _ in range(6):

        if keep.sum() < 20:
            raise RuntimeError(
                "Too few retained pixels during background fitting."
            )

        design = np.column_stack(
            [
                np.ones(
                    keep.sum()
                ),
                xx[keep],
                yy[keep],
            ]
        )

        coefficients, *_ = np.linalg.lstsq(
            design,
            zz[keep],
            rcond=None,
        )

        residual = (
            zz
            - (
                coefficients[0]
                + coefficients[1] * xx
                + coefficients[2] * yy
            )
        )

        median = np.median(
            residual[keep]
        )

        mad = (
            1.4826
            * np.median(
                np.abs(
                    residual[keep]
                    - median
                )
            )
            + 1e-12
        )

        new_keep = (
            np.abs(
                residual
                - median
            )
            < 3.5 * mad
        )

        if np.array_equal(
            new_keep,
            keep,
        ):
            break

        keep = new_keep

    return coefficients


def sigma_clipped_profile(
    data,
    unc,
    rell,
    rmax,
    pixscale_arcsec,
):
    edges = np.arange(
        0.0,
        (
            W1_PROFILE_OUTER_RMAX
            * float(rmax)
            + PROFILE_BIN_ARCSEC
        ),
        PROFILE_BIN_ARCSEC,
    )

    rows = []

    psf_area_pixels = (
        math.pi
        * (
            WISE_W1_PSF_FWHM_ARCSEC
            / 2.0
        ) ** 2
        / float(
            pixscale_arcsec
        ) ** 2
    )

    for lower, upper in zip(
        edges[:-1],
        edges[1:],
    ):

        selected = (
            (rell >= lower)
            & (rell < upper)
            & np.isfinite(data)
            & np.isfinite(unc)
            & (unc > 0)
        )

        values = data[selected]
        uncertainties = unc[selected]

        if len(values) < 15:
            continue

        clipped = sigma_clip(
            values,
            sigma=3.0,
            maxiters=5,
        )

        if np.isscalar(
            clipped.mask
        ):
            good = np.ones(
                len(values),
                dtype=bool,
            )

            if bool(
                clipped.mask
            ):
                good[:] = False

        else:
            good = ~np.asarray(
                clipped.mask,
                dtype=bool,
            )

        if good.sum() < 10:
            continue

        retained_values = values[
            good
        ]

        retained_uncertainties = (
            uncertainties[good]
        )

        median = float(
            np.median(
                retained_values
            )
        )

        mad = float(
            1.4826
            * np.median(
                np.abs(
                    retained_values
                    - median
                )
            )
        )

        n_effective = max(
            good.sum()
            / max(
                psf_area_pixels,
                1.0,
            ),
            1.0,
        )

        error = max(
            float(
                np.median(
                    retained_uncertainties
                )
                / np.sqrt(
                    n_effective
                )
            ),
            mad
            / np.sqrt(
                n_effective
            ),
            1e-6,
        )

        rows.append(
            {
                "radius_arcsec": (
                    0.5
                    * (
                        lower
                        + upper
                    )
                ),
                "intensity_DN_per_pixel": median,
                "e_intensity_DN_per_pixel": error,
                "N_pixels": int(
                    good.sum()
                ),
                "N_effective": float(
                    n_effective
                ),
            }
        )

    return pd.DataFrame(
        rows
    )


# ===================================================================
# Disk and bulge profile fitting
# ===================================================================

def bn_sersic(n):
    return (
        2.0 * n
        - 1.0 / 3.0
        + 0.009876 / n
    )


def disk_model(
    radius,
    parameters,
):
    background, log_i0, log_h = (
        parameters
    )

    return (
        background
        + np.exp(log_i0)
        * np.exp(
            -radius
            / np.exp(log_h)
        )
    )


def disk_bulge_model(
    radius,
    parameters,
):
    (
        background,
        log_i0,
        log_h,
        log_ie,
        log_re,
        n,
    ) = parameters

    disk = (
        np.exp(log_i0)
        * np.exp(
            -radius
            / np.exp(log_h)
        )
    )

    re_value = np.exp(
        log_re
    )

    bulge = (
        np.exp(log_ie)
        * np.exp(
            -bn_sersic(n)
            * (
                (
                    np.maximum(
                        radius,
                        1e-4,
                    )
                    / re_value
                ) ** (1.0 / n)
                - 1.0
            )
        )
    )

    return (
        background
        + disk
        + bulge
    )


def fit_w1_profile(
    profile,
    rmax,
):
    fit_table = profile[
        np.isfinite(
            profile[
                "intensity_DN_per_pixel"
            ]
        )
    ].copy()

    fit_table = fit_table[
        fit_table[
            "radius_arcsec"
        ]
        <= (
            W1_PROFILE_OUTER_RMAX
            * float(rmax)
        )
    ]

    if len(
        fit_table
    ) < MIN_PROFILE_BINS:
        raise RuntimeError(
            "Too few W1 profile bins."
        )

    radius = fit_table[
        "radius_arcsec"
    ].to_numpy(
        float
    )

    intensity = fit_table[
        "intensity_DN_per_pixel"
    ].to_numpy(
        float
    )

    error = fit_table[
        "e_intensity_DN_per_pixel"
    ].to_numpy(
        float
    )

    valid = (
        np.isfinite(radius)
        & np.isfinite(intensity)
        & np.isfinite(error)
        & (error > 0)
    )

    radius = radius[valid]
    intensity = intensity[valid]
    error = error[valid]

    if len(
        radius
    ) < MIN_PROFILE_BINS:
        raise RuntimeError(
            "Too few valid W1 profile bins."
        )

    positive = intensity[
        intensity > 0
    ]

    initial_intensity = (
        np.percentile(
            positive,
            80,
        )
        if len(positive)
        else max(
            np.std(
                intensity
            ),
            1.0,
        )
    )

    initial_scale = max(
        0.25 * float(rmax),
        PROFILE_BIN_ARCSEC,
    )

    background_scale = max(
        np.median(
            error
        ),
        1e-4,
    )

    disk_initial = np.array(
        [
            0.0,
            np.log(
                initial_intensity
            ),
            np.log(
                initial_scale
            ),
        ]
    )

    disk_bounds = (
        [
            -10
            * background_scale,
            np.log(1e-8),
            np.log(
                PROFILE_BIN_ARCSEC
                / 2.0
            ),
        ],
        [
            10
            * background_scale,
            np.log(
                max(
                    initial_intensity
                    * 100,
                    1.0,
                )
            ),
            np.log(
                2.0
                * float(rmax)
            ),
        ],
    )

    disk_fit = least_squares(
        lambda parameters: (
            intensity
            - disk_model(
                radius,
                parameters,
            )
        ) / error,
        disk_initial,
        bounds=disk_bounds,
        loss="soft_l1",
        f_scale=2.0,
        max_nfev=5000,
    )

    disk_residual = (
        intensity
        - disk_model(
            radius,
            disk_fit.x,
        )
    ) / error

    disk_chi2 = float(
        np.sum(
            disk_residual**2
        )
    )

    disk_bic = (
        disk_chi2
        + len(
            disk_fit.x
        )
        * np.log(
            len(radius)
        )
    )

    disk_bulge_initial = np.array(
        [
            disk_fit.x[0],
            disk_fit.x[1],
            disk_fit.x[2],
            np.log(
                max(
                    initial_intensity
                    * 0.5,
                    1e-6,
                )
            ),
            np.log(
                max(
                    0.08
                    * float(rmax),
                    PROFILE_BIN_ARCSEC
                    / 2.0,
                )
            ),
            2.0,
        ]
    )

    disk_bulge_bounds = (
        [
            -10
            * background_scale,
            np.log(1e-8),
            np.log(
                PROFILE_BIN_ARCSEC
                / 2.0
            ),
            np.log(1e-8),
            np.log(
                PROFILE_BIN_ARCSEC
                / 3.0
            ),
            0.5,
        ],
        [
            10
            * background_scale,
            np.log(
                max(
                    initial_intensity
                    * 100,
                    1.0,
                )
            ),
            np.log(
                2.0
                * float(rmax)
            ),
            np.log(
                max(
                    initial_intensity
                    * 100,
                    1.0,
                )
            ),
            np.log(
                0.5
                * float(rmax)
            ),
            6.0,
        ],
    )

    disk_bulge_fit = least_squares(
        lambda parameters: (
            intensity
            - disk_bulge_model(
                radius,
                parameters,
            )
        ) / error,
        disk_bulge_initial,
        bounds=disk_bulge_bounds,
        loss="soft_l1",
        f_scale=2.0,
        max_nfev=10000,
    )

    disk_bulge_residual = (
        intensity
        - disk_bulge_model(
            radius,
            disk_bulge_fit.x,
        )
    ) / error

    disk_bulge_chi2 = float(
        np.sum(
            disk_bulge_residual**2
        )
    )

    disk_bulge_bic = (
        disk_bulge_chi2
        + len(
            disk_bulge_fit.x
        )
        * np.log(
            len(radius)
        )
    )

    delta_bic = (
        disk_bic
        - disk_bulge_bic
    )

    use_bulge = (
        delta_bic
        > BULGE_DELTA_BIC_THRESHOLD
    )

    return {
        "disk_parameters": (
            disk_fit.x
        ),
        "disk_chi2": (
            disk_chi2
        ),
        "disk_bic": (
            disk_bic
        ),
        "disk_bulge_parameters": (
            disk_bulge_fit.x
        ),
        "disk_bulge_chi2": (
            disk_bulge_chi2
        ),
        "disk_bulge_bic": (
            disk_bulge_bic
        ),
        "delta_bic_disk_minus_diskbulge": (
            delta_bic
        ),
        "use_bulge": bool(
            use_bulge
        ),
        "N_fit_bins": int(
            len(radius)
        ),
    }


# ===================================================================
# Axisymmetric gravity functions
# ===================================================================

def ring_disk_v2(
    r_eval,
    r_grid,
    sigma_msun_pc2,
    z0_kpc,
):
    r_eval = np.asarray(
        r_eval,
        dtype=float,
    )

    r_grid = np.asarray(
        r_grid,
        dtype=float,
    )

    sigma = np.asarray(
        sigma_msun_pc2,
        dtype=float,
    )

    if (
        len(r_grid) < 3
        or not np.isfinite(
            sigma
        ).any()
        or np.nanmax(
            sigma
        ) <= 0
    ):
        return np.zeros_like(
            r_eval
        )

    edges = np.empty(
        len(r_grid) + 1
    )

    edges[1:-1] = (
        0.5
        * (
            r_grid[:-1]
            + r_grid[1:]
        )
    )

    edges[0] = max(
        0.0,
        r_grid[0]
        - 0.5
        * (
            r_grid[1]
            - r_grid[0]
        ),
    )

    edges[-1] = (
        r_grid[-1]
        + 0.5
        * (
            r_grid[-1]
            - r_grid[-2]
        )
    )

    dr = np.diff(
        edges
    )

    ring_mass = (
        2.0
        * np.pi
        * r_grid
        * dr
        * sigma
        * 1e6
    )

    phi = np.linspace(
        0.0,
        2.0 * np.pi,
        192,
        endpoint=False,
    )

    cosine_phi = np.cos(
        phi
    )

    result = np.zeros_like(
        r_eval
    )

    for index, radius in enumerate(
        r_eval
    ):

        if radius <= 0:
            continue

        ring_radius = r_grid[
            :,
            None,
        ]

        denominator = (
            radius**2
            + ring_radius**2
            - (
                2.0
                * radius
                * ring_radius
                * cosine_phi[
                    None,
                    :,
                ]
            )
            + float(
                z0_kpc
            ) ** 2
        ) ** 1.5

        kernel = np.mean(
            (
                radius
                - ring_radius
                * cosine_phi[
                    None,
                    :,
                ]
            )
            / denominator,
            axis=1,
        )

        radial_acceleration = (
            G_KPC_KMS2_MSUN
            * np.sum(
                ring_mass
                * kernel
            )
        )

        result[index] = max(
            radius
            * radial_acceleration,
            0.0,
        )

    return result


def bulge_v2_sersic(
    r_eval,
    ie_lsun_pc2,
    re_kpc,
    n,
    ml=1.0,
):
    r_eval = np.asarray(
        r_eval,
        dtype=float,
    )

    if (
        not np.isfinite(
            ie_lsun_pc2
        )
        or not np.isfinite(
            re_kpc
        )
        or not np.isfinite(n)
        or ie_lsun_pc2 <= 0
        or re_kpc <= 0
    ):
        return np.zeros_like(
            r_eval
        )

    b_value = bn_sersic(
        n
    )

    total_luminosity = (
        2.0
        * np.pi
        * ie_lsun_pc2
        * 1e6
        * re_kpc**2
        * n
        * np.exp(
            b_value
        )
        * b_value ** (
            -2.0 * n
        )
        * gamma(
            2.0 * n
        )
    )

    p_value = (
        1.0
        - 0.6097 / n
        + 0.05463 / n**2
    )

    shape_parameter = (
        3.0
        - p_value
    ) * n

    x_value = (
        b_value
        * (
            np.maximum(
                r_eval,
                1e-6,
            )
            / re_kpc
        ) ** (
            1.0 / n
        )
    )

    enclosed_mass = (
        float(ml)
        * total_luminosity
        * gammainc(
            shape_parameter,
            x_value,
        )
    )

    return (
        G_KPC_KMS2_MSUN
        * enclosed_mass
        / np.maximum(
            r_eval,
            1e-6,
        )
    )


def luminosity_surface_density_per_DN(
    header,
    distance_mpc,
    pixscale_arcsec,
):
    if "MAGZP" not in header:
        raise RuntimeError(
            "AllWISE intensity FITS header lacks MAGZP."
        )

    magzp = float(
        header["MAGZP"]
    )

    distance_pc = (
        float(
            distance_mpc
        )
        * 1e6
    )

    distance_modulus = (
        5.0
        * np.log10(
            distance_pc
            / 10.0
        )
    )

    luminosity_one_dn = 10.0 ** (
        -0.4
        * (
            magzp
            - distance_modulus
            - WISE_W1_SOLAR_MAG_VEGA
        )
    )

    projected_pixel_size_pc = (
        distance_pc
        * np.deg2rad(
            float(
                pixscale_arcsec
            )
            / 3600.0
        )
    )

    projected_pixel_area_pc2 = (
        projected_pixel_size_pc**2
    )

    return (
        luminosity_one_dn
        / projected_pixel_area_pc2
    )


# ===================================================================
# Process all targets
# ===================================================================

profile_rows = []
fit_rows = []
baryonic_rows = []
failure_rows = []

acquisition_by_id = acquisition.set_index(
    "source_product_id"
)


for _, target in targets.iterrows():

    source_id = target[
        "source_product_id"
    ]

    acquisition_row = acquisition_by_id.loc[
        source_id
    ]

    if (
        acquisition_row["status"]
        != "success"
    ):
        failure_rows.append(
            {
                "source_product_id": source_id,
                "galaxy": target[
                    "galaxy"
                ],
                "sample": target[
                    "sample"
                ],
                "stage": "acquisition",
                "error": acquisition_row[
                    "error"
                ],
            }
        )

        fit_rows.append(
            {
                "source_product_id": source_id,
                "galaxy": target[
                    "galaxy"
                ],
                "sample": target[
                    "sample"
                ],
                "status": "failed",
                "error": acquisition_row[
                    "error"
                ],
                "rotation_values_used": False,
            }
        )

        continue

    try:
        intensity_path = (
            W1
            / acquisition_row[
                "intensity_file"
            ]
        )

        uncertainty_path = (
            W1
            / acquisition_row[
                "uncertainty_file"
            ]
        )

        with fits.open(
            intensity_path,
            memmap=False,
        ) as hdul:
            image = np.asarray(
                hdul[0].data,
                dtype=float,
            )

            header = hdul[
                0
            ].header.copy()

        with fits.open(
            uncertainty_path,
            memmap=False,
        ) as hdul:
            uncertainty = np.asarray(
                hdul[0].data,
                dtype=float,
            )

        image = np.squeeze(
            image
        )

        uncertainty = np.squeeze(
            uncertainty
        )

        if image.ndim != 2:
            raise RuntimeError(
                "Unexpected W1 intensity image shape: "
                f"{image.shape}"
            )

        if uncertainty.ndim != 2:
            raise RuntimeError(
                "Unexpected W1 uncertainty image shape: "
                f"{uncertainty.shape}"
            )

        if image.shape != uncertainty.shape:
            raise RuntimeError(
                "Intensity and uncertainty shapes differ: "
                f"{image.shape} versus {uncertainty.shape}."
            )

        wcs = WCS(
            header
        ).celestial

        (
            east,
            north,
            centre_x,
            centre_y,
        ) = local_tangent_offsets_arcsec(
            wcs=wcs,
            shape=image.shape,
            ra=target[
                "w1_ra_deg"
            ],
            dec=target[
                "w1_dec_deg"
            ],
        )

        elliptical_radii = elliptical_radius(
            east=east,
            north=north,
            pa_deg=target[
                "position_angle_deg"
            ],
            inc_deg=target[
                "inclination_deg"
            ],
        )

        background_mask = (
            (
                elliptical_radii
                >= (
                    W1_BACKGROUND_INNER_RMAX
                    * float(
                        target[
                            "rmax_arcsec"
                        ]
                    )
                )
            )
            & (
                elliptical_radii
                <= (
                    W1_BACKGROUND_OUTER_RMAX
                    * float(
                        target[
                            "rmax_arcsec"
                        ]
                    )
                )
            )
        )

        background_coefficients = (
            robust_background_plane(
                data=image,
                east=east,
                north=north,
                mask=background_mask,
            )
        )

        background = (
            background_coefficients[0]
            + background_coefficients[1]
            * east
            + background_coefficients[2]
            * north
        )

        background_subtracted_image = (
            image
            - background
        )

        pixel_scale_arcsec = float(
            np.hypot(
                east[0, 1]
                - east[0, 0],
                north[0, 1]
                - north[0, 0],
            )
        )

        if (
            not np.isfinite(
                pixel_scale_arcsec
            )
            or pixel_scale_arcsec <= 0
        ):
            raise RuntimeError(
                "Invalid W1 pixel scale."
            )

        profile = sigma_clipped_profile(
            data=background_subtracted_image,
            unc=uncertainty,
            rell=elliptical_radii,
            rmax=target[
                "rmax_arcsec"
            ],
            pixscale_arcsec=(
                pixel_scale_arcsec
            ),
        )

        profile_fit = fit_w1_profile(
            profile=profile,
            rmax=target[
                "rmax_arcsec"
            ],
        )

        for profile_row in profile.to_dict(
            orient="records"
        ):
            profile_rows.append(
                {
                    "source_product_id": source_id,
                    "galaxy": target[
                        "galaxy"
                    ],
                    "sample": target[
                        "sample"
                    ],
                    **profile_row,
                }
            )

        if profile_fit[
            "use_bulge"
        ]:
            (
                fitted_background,
                log_disk_i0,
                log_disk_h,
                log_bulge_ie,
                log_bulge_re,
                bulge_n,
            ) = profile_fit[
                "disk_bulge_parameters"
            ]

            bulge_ie_dn = float(
                np.exp(
                    log_bulge_ie
                )
            )

            bulge_re_arcsec = float(
                np.exp(
                    log_bulge_re
                )
            )

            bulge_n = float(
                bulge_n
            )

        else:
            (
                fitted_background,
                log_disk_i0,
                log_disk_h,
            ) = profile_fit[
                "disk_parameters"
            ]

            bulge_ie_dn = 0.0
            bulge_re_arcsec = np.nan
            bulge_n = np.nan

        disk_i0_dn = float(
            np.exp(
                log_disk_i0
            )
        )

        disk_h_arcsec = float(
            np.exp(
                log_disk_h
            )
        )

        distance_mpc = float(
            target[
                "distance_mpc"
            ]
        )

        kpc_per_arcsec = (
            distance_mpc
            * 1000.0
            * np.deg2rad(
                1.0
                / 3600.0
            )
        )

        disk_h_kpc = (
            disk_h_arcsec
            * kpc_per_arcsec
        )

        bulge_re_kpc = (
            bulge_re_arcsec
            * kpc_per_arcsec
            if np.isfinite(
                bulge_re_arcsec
            )
            else np.nan
        )

        luminosity_per_dn = (
            luminosity_surface_density_per_DN(
                header=header,
                distance_mpc=distance_mpc,
                pixscale_arcsec=(
                    pixel_scale_arcsec
                ),
            )
        )

        cos_inclination = max(
            np.cos(
                np.deg2rad(
                    float(
                        target[
                            "inclination_deg"
                        ]
                    )
                )
            ),
            0.15,
        )

        disk_sigma0_lsun_pc2 = (
            disk_i0_dn
            * luminosity_per_dn
            * cos_inclination
        )

        bulge_ie_lsun_pc2 = (
            bulge_ie_dn
            * luminosity_per_dn
        )

        kinematic_row = get_kinematic_row(
            target
        )

        radius_arcsec = parse_array(
            kinematic_row[
                "Rad"
            ]
        )

        if len(
            radius_arcsec
        ) == 0:
            raise RuntimeError(
                "No valid Rad array was found."
            )

        radius_kpc = (
            radius_arcsec
            * kpc_per_arcsec
        )

        gas_radius_arcsec = parse_array(
            kinematic_row[
                "Rad_SD"
            ]
        )

        gas_surface_density_hi = parse_array(
            kinematic_row.get(
                "SD_FO_model",
                kinematic_row.get(
                    "SD_model"
                ),
            )
        )

        gas_surface_density_error = parse_array(
            kinematic_row.get(
                "e_SD_FO_model_inc",
                kinematic_row.get(
                    "e_SD_model"
                ),
            )
        )

        n_gas = min(
            len(
                gas_radius_arcsec
            ),
            len(
                gas_surface_density_hi
            ),
            len(
                gas_surface_density_error
            ),
        )

        gas_radius_kpc = (
            gas_radius_arcsec[
                :n_gas
            ]
            * kpc_per_arcsec
        )

        gas_surface_density = (
            HELIUM_FACTOR
            * np.clip(
                gas_surface_density_hi[
                    :n_gas
                ],
                0.0,
                None,
            )
        )

        gas_valid = (
            np.isfinite(
                gas_radius_kpc
            )
            & np.isfinite(
                gas_surface_density
            )
            & (
                gas_radius_kpc
                >= 0
            )
        )

        gas_radius_kpc = (
            gas_radius_kpc[
                gas_valid
            ]
        )

        gas_surface_density = (
            gas_surface_density[
                gas_valid
            ]
        )

        if len(
            gas_radius_kpc
        ) > 1:
            gas_order = np.argsort(
                gas_radius_kpc
            )

            gas_radius_kpc = (
                gas_radius_kpc[
                    gas_order
                ]
            )

            gas_surface_density = (
                gas_surface_density[
                    gas_order
                ]
            )

        source_outer_kpc = max(
            float(
                np.nanmax(
                    radius_kpc
                )
                * 1.5
            ),
            (
                float(
                    np.nanmax(
                        gas_radius_kpc
                    )
                    * 1.2
                )
                if len(
                    gas_radius_kpc
                )
                else 0.0
            ),
            float(
                6.0
                * disk_h_kpc
            ),
            0.5,
        )

        source_grid_kpc = np.linspace(
            max(
                source_outer_kpc
                / 400.0,
                0.002,
            ),
            source_outer_kpc,
            400,
        )

        stellar_surface_density = (
            disk_sigma0_lsun_pc2
            * np.exp(
                -source_grid_kpc
                / max(
                    disk_h_kpc,
                    1e-6,
                )
            )
        )

        stellar_scale_height_kpc = max(
            MIN_STELLAR_SCALE_HEIGHT_KPC,
            (
                DISK_THICKNESS_OVER_SCALE_LENGTH
                * disk_h_kpc
            ),
        )

        vdisk2_unit = ring_disk_v2(
            r_eval=radius_kpc,
            r_grid=source_grid_kpc,
            sigma_msun_pc2=(
                stellar_surface_density
            ),
            z0_kpc=(
                stellar_scale_height_kpc
            ),
        )

        if len(
            gas_radius_kpc
        ) >= 2:
            gas_surface_density_grid = np.interp(
                source_grid_kpc,
                gas_radius_kpc,
                gas_surface_density,
                left=gas_surface_density[
                    0
                ],
                right=0.0,
            )

            vgas2 = ring_disk_v2(
                r_eval=radius_kpc,
                r_grid=source_grid_kpc,
                sigma_msun_pc2=(
                    gas_surface_density_grid
                ),
                z0_kpc=(
                    GAS_SCALE_HEIGHT_KPC
                ),
            )

        else:
            vgas2 = np.zeros_like(
                radius_kpc
            )

        if profile_fit[
            "use_bulge"
        ]:
            vbulge2_unit = (
                bulge_v2_sersic(
                    r_eval=radius_kpc,
                    ie_lsun_pc2=(
                        bulge_ie_lsun_pc2
                    ),
                    re_kpc=(
                        bulge_re_kpc
                    ),
                    n=bulge_n,
                    ml=1.0,
                )
            )

        else:
            vbulge2_unit = np.zeros_like(
                radius_kpc
            )

        vbar2_primary = (
            vgas2
            + PRIMARY_YDISK
            * vdisk2_unit
            + PRIMARY_YBULGE
            * vbulge2_unit
        )

        gbar_primary = (
            vbar2_primary
            * 1e6
            / np.maximum(
                radius_kpc
                * KPC_TO_M,
                1e-30,
            )
        )

        for ring_index in range(
            len(
                radius_arcsec
            )
        ):
            baryonic_rows.append(
                {
                    "source_product_id": source_id,
                    "galaxy": target[
                        "galaxy"
                    ],
                    "sample": target[
                        "sample"
                    ],
                    "ring_index": int(
                        ring_index
                    ),
                    "radius_arcsec": float(
                        radius_arcsec[
                            ring_index
                        ]
                    ),
                    "radius_kpc": float(
                        radius_kpc[
                            ring_index
                        ]
                    ),
                    "vgas_kms": float(
                        np.sqrt(
                            max(
                                vgas2[
                                    ring_index
                                ],
                                0.0,
                            )
                        )
                    ),
                    "vdisk_unitml_kms": float(
                        np.sqrt(
                            max(
                                vdisk2_unit[
                                    ring_index
                                ],
                                0.0,
                            )
                        )
                    ),
                    "vbulge_unitml_kms": float(
                        np.sqrt(
                            max(
                                vbulge2_unit[
                                    ring_index
                                ],
                                0.0,
                            )
                        )
                    ),
                    "Ydisk_primary": float(
                        PRIMARY_YDISK
                    ),
                    "Ybulge_primary": float(
                        PRIMARY_YBULGE
                    ),
                    "vbar_primary_kms": float(
                        np.sqrt(
                            max(
                                vbar2_primary[
                                    ring_index
                                ],
                                0.0,
                            )
                        )
                    ),
                    "gbar_primary_m_s2": float(
                        gbar_primary[
                            ring_index
                        ]
                    ),
                }
            )

        fit_rows.append(
            {
                "source_product_id": source_id,
                "galaxy": target[
                    "galaxy"
                ],
                "sample": target[
                    "sample"
                ],
                "status": "success",
                "error": "",
                "distance_mpc": distance_mpc,
                "MAGZP": float(
                    header[
                        "MAGZP"
                    ]
                ),
                "pixel_scale_arcsec": (
                    pixel_scale_arcsec
                ),
                "centre_x_pixel": centre_x,
                "centre_y_pixel": centre_y,
                "background_plane_c0": float(
                    background_coefficients[
                        0
                    ]
                ),
                "background_plane_c_east": float(
                    background_coefficients[
                        1
                    ]
                ),
                "background_plane_c_north": float(
                    background_coefficients[
                        2
                    ]
                ),
                "N_profile_bins": int(
                    profile_fit[
                        "N_fit_bins"
                    ]
                ),
                "disk_bic": float(
                    profile_fit[
                        "disk_bic"
                    ]
                ),
                "disk_bulge_bic": float(
                    profile_fit[
                        "disk_bulge_bic"
                    ]
                ),
                "delta_bic_disk_minus_diskbulge": float(
                    profile_fit[
                        "delta_bic_disk_minus_diskbulge"
                    ]
                ),
                "bulge_included": bool(
                    profile_fit[
                        "use_bulge"
                    ]
                ),
                "disk_I0_DN_per_pixel": (
                    disk_i0_dn
                ),
                "disk_scale_arcsec": (
                    disk_h_arcsec
                ),
                "disk_scale_kpc": (
                    disk_h_kpc
                ),
                "disk_sigma0_Lsun_pc2": (
                    disk_sigma0_lsun_pc2
                ),
                "stellar_scale_height_kpc": (
                    stellar_scale_height_kpc
                ),
                "bulge_Ie_DN_per_pixel": (
                    bulge_ie_dn
                ),
                "bulge_Re_arcsec": (
                    bulge_re_arcsec
                ),
                "bulge_Re_kpc": (
                    bulge_re_kpc
                ),
                "bulge_n": bulge_n,
                "bulge_Ie_Lsun_pc2": (
                    bulge_ie_lsun_pc2
                ),
                "rotation_values_used": False,
            }
        )

        print(
            target["sample"],
            target["galaxy"],
            "processed",
            flush=True,
        )

    except Exception as exc:
        error_text = (
            f"{type(exc).__name__}: {exc}"
        )

        failure_rows.append(
            {
                "source_product_id": source_id,
                "galaxy": target[
                    "galaxy"
                ],
                "sample": target[
                    "sample"
                ],
                "stage": (
                    "profile_or_gravity"
                ),
                "error": error_text,
            }
        )

        fit_rows.append(
            {
                "source_product_id": source_id,
                "galaxy": target[
                    "galaxy"
                ],
                "sample": target[
                    "sample"
                ],
                "status": "failed",
                "error": error_text,
                "rotation_values_used": False,
            }
        )

        print(
            target["sample"],
            target["galaxy"],
            "FAILED",
            error_text,
            flush=True,
        )


# ===================================================================
# Save unscored E2 products
# ===================================================================

profiles = pd.DataFrame(
    profile_rows,
    columns=[
        "source_product_id",
        "galaxy",
        "sample",
        "radius_arcsec",
        "intensity_DN_per_pixel",
        "e_intensity_DN_per_pixel",
        "N_pixels",
        "N_effective",
    ],
)

fits_audit = pd.DataFrame(
    fit_rows,
    columns=[
        "source_product_id",
        "galaxy",
        "sample",
        "status",
        "error",
        "distance_mpc",
        "MAGZP",
        "pixel_scale_arcsec",
        "centre_x_pixel",
        "centre_y_pixel",
        "background_plane_c0",
        "background_plane_c_east",
        "background_plane_c_north",
        "N_profile_bins",
        "disk_bic",
        "disk_bulge_bic",
        "delta_bic_disk_minus_diskbulge",
        "bulge_included",
        "disk_I0_DN_per_pixel",
        "disk_scale_arcsec",
        "disk_scale_kpc",
        "disk_sigma0_Lsun_pc2",
        "stellar_scale_height_kpc",
        "bulge_Ie_DN_per_pixel",
        "bulge_Re_arcsec",
        "bulge_Re_kpc",
        "bulge_n",
        "bulge_Ie_Lsun_pc2",
        "rotation_values_used",
    ],
)

baryonic = pd.DataFrame(
    baryonic_rows,
    columns=[
        "source_product_id",
        "galaxy",
        "sample",
        "ring_index",
        "radius_arcsec",
        "radius_kpc",
        "vgas_kms",
        "vdisk_unitml_kms",
        "vbulge_unitml_kms",
        "Ydisk_primary",
        "Ybulge_primary",
        "vbar_primary_kms",
        "gbar_primary_m_s2",
    ],
)

failures = pd.DataFrame(
    failure_rows,
    columns=[
        "source_product_id",
        "galaxy",
        "sample",
        "stage",
        "error",
    ],
)

profiles.to_csv(
    TABLES
    / "allwise_W1_elliptical_profiles.csv",
    index=False,
)

fits_audit.to_csv(
    AUDIT
    / "allwise_W1_disk_bulge_fit_audit.csv",
    index=False,
)

baryonic.to_csv(
    TABLES
    / "UNSCORED_wallaby_baryonic_rotation_ledger.csv",
    index=False,
)

failures.to_csv(
    AUDIT
    / "E2_processing_failures.csv",
    index=False,
)

success_summary = (
    fits_audit
    .groupby(
        [
            "sample",
            "status",
        ]
    )
    .size()
    .reset_index(
        name="N"
    )
)

display(
    success_summary
)

successful_reconstructions = int(
    (
        fits_audit[
            "status"
        ]
        == "success"
    ).sum()
)

failed_reconstructions = int(
    (
        fits_audit[
            "status"
        ]
        == "failed"
    ).sum()
)

print(
    "Successful baryonic reconstructions:",
    successful_reconstructions,
)

print(
    "Failed baryonic reconstructions:",
    failed_reconstructions,
)

print(
    "Galaxies represented in baryonic ledger:",
    int(
        baryonic[
            "source_product_id"
        ].nunique()
    )
    if len(
        baryonic
    )
    else 0,
)

print(
    "Baryonic ledger radial rows:",
    len(
        baryonic
    ),
)

print(
    "No observed Vrot values were read or scored."
)

Exact-release kinematic matching enabled.
W1 profile centres corrected: 1
Observed Vrot values accessed: False
DR2_phase2_primary WALLABY J094524-480828 processed
DR2_phase2_primary WALLABY J100916-431959 processed
DR2_phase2_primary WALLABY J101655-485238 processed
DR2_phase2_primary WALLABY J125548+041805 processed
DR2_phase2_primary WALLABY J125549+040049 processed
DR2_phase2_primary WALLABY J125956-192430 processed
DR2_phase2_primary WALLABY J130213-145817 processed
DR2_phase2_primary WALLABY J130314-172514 processed
DR2_phase2_primary WALLABY J130415-102023 processed
DR2_phase2_primary WALLABY J130618-173039 processed
DR2_phase2_primary WALLABY J130943-163617 processed
DR2_phase2_primary WALLABY J131237-142630 processed
DR2_phase2_primary WALLABY J131658-163757 processed
DR2_phase2_primary WALLABY J132433-210813 processed
DR2_phase2_primary WALLABY J133314-160715 processed
DR1_phase1_replication WALLABY J100342-270137 processed
DR1_phase1_replication WALLABY J100426-282638 process

,sample,status,N
0,DR1_phase1_replication,failed,1
1,DR1_phase1_replication,success,23
2,DR2_phase2_primary,success,15


Successful baryonic reconstructions: 38
Failed baryonic reconstructions: 1
Galaxies represented in baryonic ledger: 38
Baryonic ledger radial rows: 484
No observed Vrot values were read or scored.


In [10]:

#@title 5. Lock E2, save W1 files to Drive, and export the light package

success = fits_audit[fits_audit["status"] == "success"]
sample_totals = targets.groupby("sample").size().to_dict()
sample_success = success.groupby("sample").size().to_dict()

dr2_success = int(sample_success.get("DR2_phase2_primary", 0))
dr1_success = int(sample_success.get("DR1_phase1_replication", 0))
dr2_total = int(sample_totals.get("DR2_phase2_primary", 0))
dr1_total = int(sample_totals.get("DR1_phase1_replication", 0))

adequacy = {
    "DR2_success": dr2_success,
    "DR2_total": dr2_total,
    "DR2_fraction": dr2_success / dr2_total if dr2_total else 0.0,
    "DR1_success": dr1_success,
    "DR1_total": dr1_total,
    "DR1_fraction": dr1_success / dr1_total if dr1_total else 0.0,
}
adequacy["pass"] = bool(
    dr2_success >= MIN_DR2_SUCCESS
    and dr1_success >= MIN_DR1_SUCCESS
    and adequacy["DR2_fraction"] >= MIN_SUCCESS_FRACTION_PER_SAMPLE
    and adequacy["DR1_fraction"] >= MIN_SUCCESS_FRACTION_PER_SAMPLE
)

baryonic_path = TABLES / "UNSCORED_wallaby_baryonic_rotation_ledger.csv"
profile_path = TABLES / "allwise_W1_elliptical_profiles.csv"
fit_path = AUDIT / "allwise_W1_disk_bulge_fit_audit.csv"
acq_path = AUDIT / "allwise_W1_acquisition_manifest.csv"

e2_registry = {
    "protocol_name": "NG22R-E2 W1 baryonic ledger lock",
    "status": (
        "BARYONIC_LEDGER_LOCKED_READY_FOR_SEALED_SCORE"
        if adequacy["pass"] else
        "BARYONIC_LEDGER_LOCKED_BUT_DATA_ADEQUACY_FAILED"
    ),
    "parent_model_freeze_sha256": PARENT_MODEL_HASH,
    "parent_E0_transfer_protocol_sha256": E0_HASH,
    "parent_E1_input_lock_sha256": EXPECTED_E1_HASH,
    "sealed_observed_rotation_ledger_sha256": sealed_hash,
    "observed_rotation_ledger_read": False,
    "E1A_amendment": {
        "manifest_issue": (
            "The recursive E1 manifest contained a stale hash for itself after rewrite; "
            "all scientific files independently verified."
        ),
        "coordinate_issue": (
            "E1 stored catalogue ra/dec. E2 uses RA_model/DEC_model where available, "
            "as specified in E0, before image acquisition."
        ),
        "amendment_table_sha256": sha256_file(
            AUDIT / "E1A_pre_score_coordinate_and_manifest_amendment.csv"
        ),
    },
    "official_distance": {
        "column": "dist_h",
        "units": "Mpc",
        "interpretation": "Local Hubble distance derived from barycentric source frequency",
    },
    "photometry": {
        "archive": "IRSA AllWISE Atlas via SIA v2",
        "collection": WISE_COLLECTION,
        "band": "W1",
        "solar_absolute_magnitude_vega": WISE_W1_SOLAR_MAG_VEGA,
        "PSF_FWHM_arcsec": WISE_W1_PSF_FWHM_ARCSEC,
        "background_annulus_Rmax": [
            W1_BACKGROUND_INNER_RMAX, W1_BACKGROUND_OUTER_RMAX
        ],
        "profile_bin_arcsec": PROFILE_BIN_ARCSEC,
        "disk_model": "exponential",
        "bulge_model": "Sersic n in [0.5,6]",
        "bulge_selection": f"Delta BIC > {BULGE_DELTA_BIC_THRESHOLD}",
    },
    "gravity": {
        "gas_surface_density": "released SD_FO_model multiplied by 1.33",
        "gas_scale_height_kpc": GAS_SCALE_HEIGHT_KPC,
        "stellar_disk_scale_height": (
            f"max({MIN_STELLAR_SCALE_HEIGHT_KPC} kpc, "
            f"{DISK_THICKNESS_OVER_SCALE_LENGTH}*disk_scale_length)"
        ),
        "axisymmetric_solver": "softened finite-thickness numerical ring integration",
        "primary_Ydisk": PRIMARY_YDISK,
        "primary_Ybulge": PRIMARY_YBULGE,
    },
    "data_adequacy_gate": {
        "minimum_success_fraction_per_sample": MIN_SUCCESS_FRACTION_PER_SAMPLE,
        "minimum_DR2_success": MIN_DR2_SUCCESS,
        "minimum_DR1_success": MIN_DR1_SUCCESS,
        "result": adequacy,
    },
    "files": {
        "baryonic_ledger": {
            "path": str(baryonic_path.relative_to(OUT)),
            "sha256": sha256_file(baryonic_path),
        },
        "W1_profiles": {
            "path": str(profile_path.relative_to(OUT)),
            "sha256": sha256_file(profile_path),
        },
        "W1_fit_audit": {
            "path": str(fit_path.relative_to(OUT)),
            "sha256": sha256_file(fit_path),
        },
        "W1_acquisition_manifest": {
            "path": str(acq_path.relative_to(OUT)),
            "sha256": sha256_file(acq_path),
        },
    },
    "next_stage": (
        "Only if data adequacy passes: merge by source_product_id/ring_index with the "
        "already hashed sealed rotation ledger and execute the frozen NG22R-D ECSM score "
        "without changing any model, baryonic, quality, or sample rule."
    ),
}
canonical = json.dumps(e2_registry, sort_keys=True, separators=(",", ":"))
e2_registry["E2_registry_sha256"] = hashlib.sha256(canonical.encode()).hexdigest()
(REGISTRY / "ng22r_e2_baryonic_ledger_lock_registry.json").write_text(
    json.dumps(e2_registry, indent=2)
)

# Create a manifest that deliberately excludes itself to avoid the E1 self-hash defect.
manifest_rows = []
for path in sorted(OUT.rglob("*")):
    if path.is_file() and path.name != "E2_sha256_manifest.csv":
        manifest_rows.append({
            "path": str(path.relative_to(OUT)),
            "sha256": sha256_file(path),
            "size_bytes": path.stat().st_size,
        })
pd.DataFrame(manifest_rows).to_csv(AUDIT / "E2_sha256_manifest.csv", index=False)

readme = f"""# NG22R-E2 W1 baryonic ledger lock

E2 hash: {e2_registry['E2_registry_sha256']}
Data adequacy pass: {adequacy['pass']}

DR2 successful baryonic reconstructions: {dr2_success}/{dr2_total}
DR1 successful baryonic reconstructions: {dr1_success}/{dr1_total}

The sealed observed rotation ledger was not read or scored.
"""
(OUT / "README.md").write_text(readme)

# Preserve the raw W1 images in Drive without forcing a large phone download.
drive_w1 = Path("/content/drive/MyDrive/NG22R_E2_WALLABY_W1_FITS")
if drive_w1.exists():
    shutil.rmtree(drive_w1)
shutil.copytree(W1, drive_w1)

light_zip_base = "/content/NG22R_E2_WALLABY_BARYONIC_LEDGER_LOCK_LIGHT"
light_zip = shutil.make_archive(light_zip_base, "zip", root_dir=OUT)
drive_light = Path("/content/drive/MyDrive") / Path(light_zip).name
shutil.copy2(light_zip, drive_light)

print("E2 registry SHA-256:", e2_registry["E2_registry_sha256"])
print("Data adequacy:", adequacy)
print("Light package saved to:", drive_light)
print("W1 FITS folder saved to:", drive_w1)
files.download(light_zip)

E2 registry SHA-256: 96fdb2c5e00fad5cdeca7e1cf7186399b8c956603ccaeb35209b6ec9beacea78
Data adequacy: {'DR2_success': 15, 'DR2_total': 15, 'DR2_fraction': 1.0, 'DR1_success': 23, 'DR1_total': 24, 'DR1_fraction': 0.9583333333333334, 'pass': True}
Light package saved to: /content/drive/MyDrive/NG22R_E2_WALLABY_BARYONIC_LEDGER_LOCK_LIGHT.zip
W1 FITS folder saved to: /content/drive/MyDrive/NG22R_E2_WALLABY_W1_FITS


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


## Upload after completion

Upload this file from Google Drive or the browser download:

```text
NG22R_E2_WALLABY_BARYONIC_LEDGER_LOCK_LIGHT.zip
```

The larger FITS folder remains in:

```text
My Drive/NG22R_E2_WALLABY_W1_FITS/
```

Do not manually edit the baryonic ledger or inspect the sealed E1 rotation ledger.
